In [1]:
data_path = 'data_class_renamed_CS.csv'

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================
# 1. Dataset Preparation
# ==========================

class CybersicknessDataset(Dataset):
    def __init__(self, data, target_col, scaler=None):
        # Separate features and target
        self.X = data.drop(columns=[target_col]).values.astype(np.float32)
        self.y = data[target_col].values.astype(np.int64)

        # Optionally scale features
        if scaler is not None:
            self.X = scaler.transform(self.X)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Return a (feature, label) pair as torch tensors.
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])


# Load CSV data
data_path = 'data_class_renamed_CS.csv'
df = pd.read_csv(data_path)

# Use StandardScaler on the features (exclude target 'fms')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=['fms']))
df_scaled = pd.DataFrame(X_scaled, columns=df.drop(columns=['fms']).columns)
df_scaled['fms'] = df['fms']

# Split dataset
train_df, test_df = train_test_split(df_scaled, test_size=0.25, random_state=42, stratify=df_scaled['fms'])

# Create torch Datasets and DataLoaders
train_dataset = CybersicknessDataset(train_df, target_col='fms')
test_dataset  = CybersicknessDataset(test_df,  target_col='fms')

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)


# ===========================================
# 2. Model Architecture Mimicking PrivateEye
# ===========================================

# The OpticalEncoder simulates the DONN by mapping input features to a 2D latent representation.
class OpticalEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        """
        input_dim: number of features (should be 73 in our case)
        latent_dim: total dimension of latent vector (we choose 16, to be reshaped into 4x4)
        """
        super(OpticalEncoder, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )

    def forward(self, x):
        # x: (batch_size, input_dim)
        latent = self.fc(x)  # (batch_size, latent_dim)
        # Reshape latent vector into a 2D map. For 16, we choose 4x4.
        latent_map = latent.view(-1, 1, 4, 4)  # (batch_size, channels=1, 4, 4)
        return latent_map

# The Classifier takes the masked latent (utility region) and predicts the target (cybersickness fms class).
class UtilityClassifier(nn.Module):
    def __init__(self, in_channels=1, map_size=4, utility_mask=None, num_classes=3):
        super(UtilityClassifier, self).__init__()
        self.map_size = map_size

        # Define a fixed utility mask (top-left 2x2 block)
        if utility_mask is None:
            mask = torch.zeros((map_size, map_size))
            mask[0:2, 0:2] = 1.0  # top-left 2x2 = 4 elements
            utility_mask = mask.unsqueeze(0)  # shape: (1, 4, 4)
        self.register_buffer('utility_mask', utility_mask)

        # Automatically determine input dimension from mask
        self.input_dim = int(torch.sum(self.utility_mask).item())

        # Define classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, latent_map):
        # latent_map: (batch_size, 1, 4, 4)
        masked = latent_map * self.utility_mask  # shape: (batch_size, 1, 4, 4)
        # Flatten and keep only the non-zero (unmasked) features
        masked = masked.view(masked.size(0), -1)  # (batch_size, 16)
        # Select only the masked-in (utility) positions
        mask_flat = self.utility_mask.view(-1).bool()  # (16,)
        masked_selected = masked[:, mask_flat]        # (batch_size, 4)
        return self.classifier(masked_selected)


# Combine both encoder and classifier into one model.
class PrivateEyeSim(nn.Module):
    def __init__(self, input_dim, latent_dim=16, num_classes=3):
        super(PrivateEyeSim, self).__init__()
        self.encoder = OpticalEncoder(input_dim=input_dim, latent_dim=latent_dim)
        self.classifier = UtilityClassifier(in_channels=1, map_size=4, num_classes=num_classes)

    def forward(self, x):
        latent_map = self.encoder(x)  # (batch_size, 1, 4, 4)
        logits = self.classifier(latent_map)
        return logits, latent_map

# For our dataset, the input dimension is 73 (number of features after dropping fms)
INPUT_DIM = df.drop(columns=['fms']).shape[1]
NUM_CLASSES = len(df['fms'].unique())

model = PrivateEyeSim(input_dim=INPUT_DIM, latent_dim=16, num_classes=NUM_CLASSES)
print(model)


# ==========================
# 3. Training Setup
# ==========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# (Optional) To simulate additional separation, we can add a regularizer to minimize the activation in the private region.
# In our simulation, assume the "private" region is defined as the bottom-right 2x2 of the 4x4 latent map.
def private_loss(latent_map):
    # Define mask for private region (bottom-right quadrant)
    batch_size = latent_map.size(0)
    private_mask = torch.zeros_like(latent_map)
    private_mask[:, :, 2:4, 2:4] = 1.0
    # We want the private activations to be near zero
    loss = torch.mean((latent_map * private_mask)**2)
    return loss

# Total loss: Classification loss + weight * private loss penalty
alpha = 0.1  # weight for private loss penalty

# ==========================
# 4. Training Loop
# ==========================

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits, latent_map = model(inputs)
        loss_cls = criterion(logits, labels)
        loss_priv = private_loss(latent_map)
        loss = loss_cls + alpha * loss_priv

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        total += labels.size(0)
        # Calculate accuracy
        _, preds = torch.max(logits, 1)
        correct += torch.sum(preds == labels).item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs}: Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")


# ==========================
# 5. Evaluation on Test Set
# ==========================

model.eval()
test_correct = 0
test_total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        logits, _ = model(inputs)
        _, preds = torch.max(logits, 1)
        test_correct += torch.sum(preds == labels).item()
        test_total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = test_correct / test_total
print(f"Test Accuracy: {test_acc:.4f}")

# Optionally, compute a classification report if desired.
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))


PrivateEyeSim(
  (encoder): OpticalEncoder(
    (fc): Sequential(
      (0): Linear(in_features=73, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=16, bias=True)
    )
  )
  (classifier): UtilityClassifier(
    (classifier): Sequential(
      (0): Linear(in_features=4, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=3, bias=True)
    )
  )
)
Epoch 1/20: Loss: 0.6553, Accuracy: 0.7163
Epoch 2/20: Loss: 0.3326, Accuracy: 0.8708
Epoch 3/20: Loss: 0.2611, Accuracy: 0.8962
Epoch 4/20: Loss: 0.2300, Accuracy: 0.9080
Epoch 5/20: Loss: 0.2092, Accuracy: 0.9134
Epoch 6/20: Loss: 0.1980, Accuracy: 0.9171
Epoch 7/20: Loss: 0.1838, Accuracy: 0.9234
Epoch 8/20: Loss: 0.1744, Accuracy: 0.9263
Epoch 9/20: Loss: 0.1681, Accuracy: 0.9291
Epoch 10/20: Loss: 0.1573, Accuracy: 0.9331
Epoch 11/20: Loss: 0.1544, Accuracy: 0.9358
Epoch 12/20: Loss: 0.1488, Accuracy: 0.9350
Epoch 13/20: Loss: 0.1398, Accuracy: 0.9388
Ep

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

# Load and preprocess dataset
df = pd.read_csv('data_class_renamed_CS.csv')
X = df.drop(columns=['fms']).values
y = df['fms'].values

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, stratify=y, random_state=42)

# PyTorch dataset
class CS_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(CS_Dataset(X_train, y_train), batch_size=16, shuffle=True)
test_loader  = DataLoader(CS_Dataset(X_test, y_test), batch_size=16, shuffle=False)

# DONN-like encoder (optical feature separation)
class DONNEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )

    def forward(self, x):
        x = self.encoder(x)
        return x.view(-1, 1, 4, 4)  # 4x4 latent map

# Classifier with fixed mask (only reads utility region)
class MaskedClassifier(nn.Module):
    def __init__(self, latent_shape=(1, 4, 4), num_classes=3):
        super().__init__()
        # Utility mask: top-left 2x2 of 4x4 map
        mask = torch.zeros(latent_shape)
        mask[:, 0:2, 0:2] = 1.0
        self.register_buffer('mask', mask)
        self.linear = nn.Sequential(
            nn.Linear(4, 32),  # 2x2 = 4 unmasked units
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, latent_map):
        masked = latent_map * self.mask
        masked = masked.view(masked.size(0), -1)
        return self.linear(masked[:, self.mask.view(-1).bool()])

# Full model
class PrivateEyeModel(nn.Module):
    def __init__(self, input_dim, latent_dim=16, num_classes=3):
        super().__init__()
        self.encoder = DONNEncoder(input_dim, latent_dim)
        self.classifier = MaskedClassifier(num_classes=num_classes)

    def forward(self, x):
        latent = self.encoder(x)
        return self.classifier(latent)

# Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PrivateEyeModel(input_dim=X.shape[1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Train (fewer epochs, smaller batches)
for epoch in range(5):
    model.train()
    correct, total = 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        correct += (preds.argmax(1) == yb).sum().item()
        total += yb.size(0)
    print(f"Epoch {epoch+1}: Accuracy = {correct / total:.4f}")

# Evaluate
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb).argmax(1).cpu().numpy()
        y_true.extend(yb.numpy())
        y_pred.extend(preds)

# Report
report = classification_report(y_true, y_pred, output_dict=True)
pd.DataFrame(report).transpose().to_csv("privateeye_cs_report.csv")
print("Classification report saved as 'privateeye_cs_report.csv'")


Epoch 1: Accuracy = 0.7908
Epoch 2: Accuracy = 0.8919
Epoch 3: Accuracy = 0.9050
Epoch 4: Accuracy = 0.9177
Epoch 5: Accuracy = 0.9214
Classification report saved as 'privateeye_cs_report.csv'
